In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"
X_VIEW1_PATH = os.path.join(BASE_PATH, "v4PRETRAIN_X_view1.npy")
X_VIEW2_PATH = os.path.join(BASE_PATH, "v4PRETRAIN_X_view2.npy")
Y_LABELS_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Load the 2D Weights
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "HMVCL_Encoder_2D.weights.h5")

LABEL_PERCENTAGE = 0.05 # 20% Labeled Data

# --- 2D ARCHITECTURE (Must match exactly) ---
def get_cnn_encoder_2d(input_shape=(10, 784, 1)):
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv2D(32, (2, 7), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Block 2
    x = layers.Conv2D(64, (2, 5), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # Block 3
    x = layers.Conv2D(128, (2, 3), padding='same', activation='relu')(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)

    x = layers.Flatten()(x)

    h = layers.Dense(128, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)

    return Model(inputs, [h, z], name="CNN_Encoder_2D")

def train_task_head(X_subset, y_subset, task_name, use_smote=False):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )

    if use_smote:
        try:
            smote = SMOTE(k_neighbors=1, random_state=42)
            X_train, y_train = smote.fit_resample(X_train, y_train)
        except: pass

    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

    clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss',
        n_jobs=-1, random_state=42
    )

    clf.fit(X_train, y_train, sample_weight=sample_weights)

    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")
    return y_test, y_pred

def main():
    print("--- 0. Loading 2D Data ---")

    X1 = np.load(X_VIEW1_PATH).astype('float32')
    X2 = np.load(X_VIEW2_PATH).astype('float32')

    # Reshape for 2D
    if X1.ndim == 3: X1 = X1.reshape((X1.shape[0], 10, 784, 1))
    elif X1.shape[1] == 7840: X1 = X1.reshape((X1.shape[0], 10, 784, 1))

    print(f"View 1 Shape: {X1.shape}")

    scaler = StandardScaler()
    X2 = scaler.fit_transform(X2)

    df_raw = pd.read_csv(Y_LABELS_PATH)

    # --- 1. Extract Features ---
    print("Loading 2D Encoder Weights...")
    cnn = get_cnn_encoder_2d()

    try:
        cnn.load_weights(ENCODER_WEIGHTS_PATH)
        print("Weights loaded.")
    except Exception as e:
        print(f"Error loading weights: {e}")
        return

    extractor = Model(cnn.input, cnn.get_layer("representation").output)

    print("Extracting 2D Features...")
    h_features = extractor.predict(X1, batch_size=128, verbose=1)

    X_final_all = np.concatenate([X2, h_features], axis=1)
    X_final_all = np.nan_to_num(X_final_all)

    # --- 2. Run Experiments ---
    # Reconstruct Labels (Simplified)
    app_col = next((c for c in df_raw.columns if c.endswith('application')), 'application')
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), 'category')

    labels_traffic, labels_cat, labels_app = [], [], []
    for idx, row in df_raw.iterrows():
        prefix = "VPN" if "vpn" in str(row['filename']).lower() else "NonVPN"
        t_type = "VPN" if "vpn" in str(row['filename']).lower() else "Non-VPN"
        labels_traffic.append(t_type)
        labels_cat.append(f"{prefix}_{row[cat_col]}")
        labels_app.append(f"{prefix}_{row[app_col]}")

    # Exp 1: Binary
    print("\n" + "="*30 + "\n EXPERIMENT 1: BINARY \n" + "="*30)
    y_bin = LabelEncoder().fit_transform(labels_traffic)
    y_test, y_pred = train_task_head(X_final_all, y_bin, "Binary", False)
    print(classification_report(y_test, y_pred, target_names=['Non-VPN', 'VPN'], digits=4))

    # Exp 2: Category (VPN Only)
    print("\n" + "="*30 + "\n EXPERIMENT 2: CATEGORY \n" + "="*30)
    vpn_idxs = [i for i, x in enumerate(labels_traffic) if x == 'VPN']
    X_cat = X_final_all[vpn_idxs]
    y_cat_le = LabelEncoder()
    y_cat = y_cat_le.fit_transform(np.array(labels_cat)[vpn_idxs])
    y_test, y_pred = train_task_head(X_cat, y_cat, "Category", True)
    print(classification_report(y_test, y_pred, target_names=y_cat_le.classes_, digits=4))

    # Exp 3: App (Top 6)
    print("\n" + "="*30 + "\n EXPERIMENT 3: APPS \n" + "="*30)
    target_apps = ['VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout', 'VPN_Facebook', 'VPN_YouTube', 'VPN_Email']
    app_idxs = [i for i, x in enumerate(labels_app) if x in target_apps]
    X_app = X_final_all[app_idxs]
    y_app_le = LabelEncoder()
    y_app = y_app_le.fit_transform(np.array(labels_app)[app_idxs])
    y_test, y_pred = train_task_head(X_app, y_app, "Top 6 Apps", True)
    print(classification_report(y_test, y_pred, target_names=y_app_le.classes_, digits=4))

if __name__ == "__main__":
    main()

--- 0. Loading 2D Data ---
View 1 Shape: (12555, 10, 784, 1)
Loading 2D Encoder Weights...
Weights loaded.
Extracting 2D Features...
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step

 EXPERIMENT 1: BINARY 

>>> Starting Task: Binary
    Data Shape: (12555, 265)
    >>> Binary Weighted F1: 0.8961
              precision    recall  f1-score   support

     Non-VPN     0.9590    0.9482    0.9536      9314
         VPN     0.8227    0.8554    0.8387      2614

    accuracy                         0.9279     11928
   macro avg     0.8908    0.9018    0.8961     11928
weighted avg     0.9291    0.9279    0.9284     11928


 EXPERIMENT 2: CATEGORY 

>>> Starting Task: Category
    Data Shape: (2751, 265)
    >>> Category Weighted F1: 0.6976
                   precision    recall  f1-score   support

         VPN_Chat     0.4255    0.1093    0.1739       183
        VPN_Email     0.9167    0.8730    0.8943       126
VPN_File Transfer     0.7488    0.8242    0.7847       785
          VPN_P2P     0.7755

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input Data (v4 = Reshaped 7840,1)
X_VIEW1_PATH = os.path.join(BASE_PATH, "v4PRETRAIN_X_view1.npy") # (N, 7840, 1)
X_VIEW2_PATH = os.path.join(BASE_PATH, "v4PRETRAIN_X_view2.npy") # (N, 137)
Y_LABELS_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Pre-trained Weights (From the Pure Contrastive Step)
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "v4HMVCL_Encoder_v4.weights.h5")

# Training Config
LABEL_PERCENTAGE = 0.05 # Adjust this to 0.05, 0.10, etc. for your efficiency curve

# --- 1. Architecture Definition (Must Match Pre-Training Exactly) ---
def get_cnn_encoder_v4(input_shape=(7840, 1)):
    inputs = layers.Input(shape=input_shape)

    # Block 1
    x = layers.Conv1D(32, 7, strides=2, activation='relu', padding='same')(inputs)
    x = layers.MaxPooling1D(2)(x)

    # Block 2
    x = layers.Conv1D(64, 5, strides=2, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)

    # Block 3
    x = layers.Conv1D(128, 3, strides=2, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Flatten()(x)

    # Representation Heads
    h = layers.Dense(128, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)

    return Model(inputs, [h, z], name="CNN_Encoder")

# --- 2. Generic Trainer Helper (XGBoost) ---
def train_task_head(X_subset, y_subset, task_name, use_smote=False):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    # Stratified Split (Respects the Label Percentage)
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # SMOTE (Imbalance Handling)
    if use_smote:
        class_counts = np.bincount(y_train)
        if np.min(class_counts) < 2:
            print("    ! Warning: Rare classes detected. Skipping SMOTE.")
            X_train_bal, y_train_bal = X_train, y_train
        else:
            print(f"    Applying SMOTE...")
            try:
                smote = SMOTE(k_neighbors=1, random_state=42)
                X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
            except Exception as e:
                print(f"    SMOTE Failed ({e}). Fallback to standard train.")
                X_train_bal, y_train_bal = X_train, y_train
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        n_jobs=-1,
        random_state=42
    )

    clf.fit(X_train_bal, y_train_bal, sample_weight=sample_weights)

    # Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    >>> {task_name} Weighted F1: {f1:.4f}")

    return y_test, y_pred

def main():
    # ==========================================
    # 0. LOADING & PREPROCESSING
    # ==========================================
    print("--- 0. Loading Data ---")

    # Load v4 Data
    if not os.path.exists(X_VIEW1_PATH):
        print(f"Error: {X_VIEW1_PATH} not found.")
        return

    X1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 7840, 1)
    X2 = np.load(X_VIEW2_PATH).astype('float32') # (N, 137)
    df_raw = pd.read_csv(Y_LABELS_PATH)

    print(f"Loaded View 1: {X1.shape}")
    print(f"Loaded View 2: {X2.shape}")

    # Normalize Stats
    scaler = StandardScaler()
    X2 = scaler.fit_transform(X2)

    # Reconstruct Labels (VPN Prefixing)
    print("Reconstructing Labels...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), 'application')
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), 'category')

    final_traffic, final_category, final_app = [], [], []

    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app, raw_cat = str(row[app_col]), str(row[cat_col])

        if "vpn" in fname:
            prefix, traffic_type = "VPN", "VPN"
        else:
            prefix, traffic_type = "NonVPN", "Non-VPN"

        final_traffic.append(traffic_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # ==========================================
    # 1. FEATURE EXTRACTION (The "LE-MVCL" Part)
    # ==========================================
    print(f"--- 1. Extracting Features using Pre-Trained Encoder ---")

    # Initialize Model
    cnn = get_cnn_encoder_v4(input_shape=(7840, 1))

    # Load Weights
    try:
        cnn.load_weights(ENCODER_WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"CRITICAL ERROR: Could not load weights from {ENCODER_WEIGHTS_PATH}")
        print(e)
        return

    # Create Extractor (Output = 'representation' layer, 128-dim)
    extractor = Model(inputs=cnn.input, outputs=cnn.get_layer("representation").output)

    # Predict
    print("Running CNN on Payload...")
    h_features = extractor.predict(X1, batch_size=64, verbose=1)

    # Fuse with Stats
    X_final_all = np.concatenate([X2, h_features], axis=1)

    # Clean NaNs (Just in case)
    X_final_all = np.nan_to_num(X_final_all)
    print(f"Final Fused Data Shape: {X_final_all.shape}")


    # ==========================================
    # EXPERIMENT 1: BINARY TASK (Full Data)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])

    y_test_bin, y_pred_bin = train_task_head(X_final_all, y_bin, "Binary Task", use_smote=False)
    print(classification_report(y_test_bin, y_pred_bin, target_names=le_bin.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 2: CATEGORY TASK (VPN Only)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY CLASSIFICATION")
    print("="*40)

    vpn_mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final_all[vpn_mask]
    y_cat_raw = df_labels.loc[vpn_mask, 'Category']

    print(f"VPN Samples: {len(X_cat)}")

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)

    y_test_cat, y_pred_cat = train_task_head(X_cat, y_cat, "VPN Category", use_smote=False)
    print(classification_report(y_test_cat, y_pred_cat, target_names=le_cat.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 3: APPLICATION TASK (Top 6 VPN Apps)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN APPLICATION (Top 6)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]

    app_mask = df_labels['Application'].isin(target_apps)
    X_app = X_final_all[app_mask]
    y_app_raw = df_labels.loc[app_mask, 'Application']

    print(f"Top 6 App Samples: {len(X_app)}")

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)

    y_test_app, y_pred_app = train_task_head(X_app, y_app, "VPN Top 6 Apps", use_smote=False)
    print(classification_report(y_test_app, y_pred_app, target_names=le_app.classes_, digits=4))

if __name__ == "__main__":
    main()

--- 0. Loading Data ---
Loaded View 1: (12555, 7840, 1)
Loaded View 2: (12555, 137)
Reconstructing Labels...
--- 1. Extracting Features using Pre-Trained Encoder ---
Weights loaded successfully.
Running CNN on Payload...
197/197 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step
Final Fused Data Shape: (12555, 265)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 265)
    Train: 627 | Test: 11928
    >>> Binary Task Weighted F1: 0.8960
              precision    recall  f1-score   support

     Non-VPN     0.9592    0.9477    0.9534      9314
         VPN     0.8213    0.8565    0.8386      2614

    accuracy                         0.9277     11928
   macro avg     0.8903    0.9021    0.8960     11928
weighted avg     0.9290    0.9277    0.9283     11928


 EXPERIMENT 2: VPN CATEGORY CLASSIFICATION
VPN Samples: 2751

>>> Starting Task: VPN Category
    Data Shape: (2751, 265)
    Train: 137 | Test: 2614
    >>> VPN Category Weighted F1: 0.6973
                   

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input Data (Ensure these point to your CLEAN, NEW files)
X_VIEW1_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view1.npy") # Raw Payload (N, 10, 784)
X_VIEW2_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view2.npy") # Stats (N, 137)
Y_LABELS_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Pre-trained Weights
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "v3FULL_HMVCL_Encoder.weights.h5")

# Training Config
LABEL_PERCENTAGE = 0.20 # 20% labeled data

# --- 1. Architecture Definition (MUST MATCH PRE-TRAINING) ---
def get_cnn_encoder(input_shape=(10, 784)):
    inputs = layers.Input(shape=input_shape)

    # 1. Reshape: Treat 10 packets * 784 bytes as one long sequence
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs) # (7840, 1)

    # 2. CNN Block (Exact same layers as Pre-training)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    # Flatten -> Dense
    x = layers.Flatten()(x)

    h = layers.Dense(128, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)

    # We return [h, z] to match the weights structure
    return Model(inputs, [h, z], name="CNN_Encoder")

# --- 2. Generic Trainer Helper ---
def train_task_head(X_subset, y_subset, task_name, use_smote=False):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    # Stratified Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # SMOTE (Imbalance Handling)
    if use_smote:
        class_counts = np.bincount(y_train)
        if np.min(class_counts) < 2:
            print("    ! Warning: Rare classes detected. Skipping SMOTE.")
            X_train_bal, y_train_bal = X_train, y_train
        else:
            print(f"    Applying SMOTE...")
            smote = SMOTE(k_neighbors=1, random_state=42)
            X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    # Determine Objective & Metric
    num_classes = len(np.unique(y_train_bal))
    if num_classes == 2:
        objective = 'binary:logistic'
        eval_metric = 'logloss'
    else:
        objective = 'multi:softprob'
        eval_metric = 'mlogloss'

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        eval_metric=eval_metric,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train_bal, y_train_bal, sample_weight=sample_weights)

    # Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    >>> {task_name} Macro F1: {f1:.4f}")

    return y_test, y_pred

def main():
    # ==========================================
    # 0. GLOBAL SETUP
    # ==========================================
    print("--- 0. Loading & Preprocessing ---")

    # Load Data
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, 137)

    print(f"Loaded View 1: {X_view1.shape}")
    print(f"Loaded View 2: {X_view2.shape}")

    df_raw = pd.read_csv(Y_LABELS_PATH)

    # Normalize Stats
    X_view2 = (X_view2 - np.mean(X_view2, axis=0)) / (np.std(X_view2, axis=0) + 1e-8)

    # --- LABEL RECONSTRUCTION LOGIC ---
    print("Reconstructing Labels (VPN_Prefixing)...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print("Error: Could not find 'application' or 'category' columns in CSV.")
        return

    final_traffic = []
    final_category = []
    final_app = []

    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix = "VPN"
            traffic_type = "VPN"
        else:
            prefix = "NonVPN"
            traffic_type = "Non-VPN"

        final_traffic.append(traffic_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # --- FEATURE FUSION ---
    print(f"Loading Encoder: {ENCODER_WEIGHTS_PATH}")

    # 1. Initialize Correct Model Architecture
    cnn_encoder = get_cnn_encoder(input_shape=(10, 784))

    # 2. Load Weights (This should now work perfectly)
    try:
        cnn_encoder.load_weights(ENCODER_WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"CRITICAL ERROR LOADING WEIGHTS: {e}")
        return

    # 3. Create Feature Extractor (Output is 'h', the representation)
    extractor = Model(inputs=cnn_encoder.input, outputs=cnn_encoder.outputs[0])

    print("Extracting features from Raw Payload...")
    h_features = extractor.predict(X_view1, batch_size=128, verbose=1)

    # 4. Concatenate: Stats + Learned Payload Embeddings
    X_final_all = np.concatenate([X_view2, h_features], axis=1)

    # --- CLEAN NaNs ---
    if np.isnan(X_final_all).any() or np.isinf(X_final_all).any():
        print("   ! Found NaNs/Infs in fused data. Cleaning...")
        X_final_all = np.nan_to_num(X_final_all, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Final Fused Data Shape: {X_final_all.shape}")


    # ==========================================
    # EXPERIMENT 1: BINARY TASK (Full Data)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])

    y_test_bin, y_pred_bin = train_task_head(X_final_all, y_bin, "Binary Task", use_smote=False)
    print(classification_report(y_test_bin, y_pred_bin, target_names=le_bin.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 2: CATEGORY TASK (VPN Only)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY CLASSIFICATION")
    print("="*40)

    vpn_mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final_all[vpn_mask]
    y_cat_raw = df_labels.loc[vpn_mask, 'Category']

    print(f"VPN Samples: {len(X_cat)}")

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)

    y_test_cat, y_pred_cat = train_task_head(X_cat, y_cat, "VPN Category", use_smote=False)
    print(classification_report(y_test_cat, y_pred_cat, target_names=le_cat.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 3: APPLICATION TASK (Top 6 VPN Apps)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN APPLICATION (Top 6)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]

    app_mask = df_labels['Application'].isin(target_apps)
    X_app = X_final_all[app_mask]
    y_app_raw = df_labels.loc[app_mask, 'Application']

    print(f"Top 6 App Samples: {len(X_app)}")

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)

    y_test_app, y_pred_app = train_task_head(X_app, y_app, "VPN Top 6 Apps", use_smote=False)
    print(classification_report(y_test_app, y_pred_app, target_names=le_app.classes_, digits=4))

if __name__ == "__main__":
    main()

--- 0. Loading & Preprocessing ---
Loaded View 1: (12555, 10, 784)
Loaded View 2: (12555, 137)
Reconstructing Labels (VPN_Prefixing)...
Loading Encoder: /content/drive/MyDrive/1 Skripsi/27jan/v3FULL_HMVCL_Encoder.weights.h5
Weights loaded successfully.
Extracting features from Raw Payload...
99/99 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step
Final Fused Data Shape: (12555, 265)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 265)
    Train: 2511 | Test: 10044
    >>> Binary Task Macro F1: 0.9448
              precision    recall  f1-score   support

     Non-VPN     0.9852    0.9649    0.9749      7843
         VPN     0.8836    0.9482    0.9147      2201

    accuracy                         0.9613     10044
   macro avg     0.9344    0.9566    0.9448     10044
weighted avg     0.9629    0.9613    0.9618     10044


 EXPERIMENT 2: VPN CATEGORY CLASSIFICATION
VPN Samples: 2751

>>> Starting Task: VPN Category
    Data Shape: (2751, 265)
    Train: 550 | Te

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input Data (Ensure these point to your CLEAN, NEW files)
X_VIEW1_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view1.npy") # Raw Payload (N, 10, 784)
X_VIEW2_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view2.npy") # Stats (N, 137)
Y_LABELS_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Pre-trained Weights
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "v3FULL_HMVCL_Encoder.weights.h5")

# Training Config
LABEL_PERCENTAGE = 0.20 # 20% labeled data

# --- 1. Architecture Definition (MUST MATCH PRE-TRAINING) ---
def get_cnn_encoder(input_shape=(10, 784)):
    inputs = layers.Input(shape=input_shape)

    # 1. Reshape: Treat 10 packets * 784 bytes as one long sequence
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs) # (7840, 1)

    # 2. CNN Block (Exact same layers as Pre-training)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    # Flatten -> Dense
    x = layers.Flatten()(x)

    h = layers.Dense(128, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)

    # We return [h, z] to match the weights structure
    return Model(inputs, [h, z], name="CNN_Encoder")

# --- 2. Generic Trainer Helper ---
def train_task_head(X_subset, y_subset, task_name, use_smote=False):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    # Stratified Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # SMOTE (Imbalance Handling)
    if use_smote:
        class_counts = np.bincount(y_train)
        if np.min(class_counts) < 2:
            print("    ! Warning: Rare classes detected. Skipping SMOTE.")
            X_train_bal, y_train_bal = X_train, y_train
        else:
            print(f"    Applying SMOTE...")
            smote = SMOTE(k_neighbors=1, random_state=42)
            X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    # Determine Objective & Metric
    num_classes = len(np.unique(y_train_bal))
    if num_classes == 2:
        objective = 'binary:logistic'
        eval_metric = 'logloss'
    else:
        objective = 'multi:softprob'
        eval_metric = 'mlogloss'

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        eval_metric=eval_metric,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train_bal, y_train_bal, sample_weight=sample_weights)

    # Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    >>> {task_name} Macro F1: {f1:.4f}")

    return y_test, y_pred

def main():
    # ==========================================
    # 0. GLOBAL SETUP
    # ==========================================
    print("--- 0. Loading & Preprocessing ---")

    # Load Data
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, 137)

    print(f"Loaded View 1: {X_view1.shape}")
    print(f"Loaded View 2: {X_view2.shape}")

    df_raw = pd.read_csv(Y_LABELS_PATH)

    # Normalize Stats
    X_view2 = (X_view2 - np.mean(X_view2, axis=0)) / (np.std(X_view2, axis=0) + 1e-8)

    # --- LABEL RECONSTRUCTION LOGIC ---
    print("Reconstructing Labels (VPN_Prefixing)...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print("Error: Could not find 'application' or 'category' columns in CSV.")
        return

    final_traffic = []
    final_category = []
    final_app = []

    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix = "VPN"
            traffic_type = "VPN"
        else:
            prefix = "NonVPN"
            traffic_type = "Non-VPN"

        final_traffic.append(traffic_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # --- FEATURE FUSION ---
    print(f"Loading Encoder: {ENCODER_WEIGHTS_PATH}")

    # 1. Initialize Correct Model Architecture
    cnn_encoder = get_cnn_encoder(input_shape=(10, 784))

    # 2. Load Weights (This should now work perfectly)
    try:
        cnn_encoder.load_weights(ENCODER_WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"CRITICAL ERROR LOADING WEIGHTS: {e}")
        return

    # 3. Create Feature Extractor (Output is 'h', the representation)
    extractor = Model(inputs=cnn_encoder.input, outputs=cnn_encoder.outputs[0])

    print("Extracting features from Raw Payload...")
    h_features = extractor.predict(X_view1, batch_size=128, verbose=1)

    # 4. Concatenate: Stats + Learned Payload Embeddings
    X_final_all = np.concatenate([X_view2, h_features], axis=1)

    # --- CLEAN NaNs ---
    if np.isnan(X_final_all).any() or np.isinf(X_final_all).any():
        print("   ! Found NaNs/Infs in fused data. Cleaning...")
        X_final_all = np.nan_to_num(X_final_all, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Final Fused Data Shape: {X_final_all.shape}")


    # ==========================================
    # EXPERIMENT 1: BINARY TASK (Full Data)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])

    y_test_bin, y_pred_bin = train_task_head(X_final_all, y_bin, "Binary Task", use_smote=False)
    print(classification_report(y_test_bin, y_pred_bin, target_names=le_bin.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 2: CATEGORY TASK (VPN Only)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY CLASSIFICATION")
    print("="*40)

    vpn_mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final_all[vpn_mask]
    y_cat_raw = df_labels.loc[vpn_mask, 'Category']

    print(f"VPN Samples: {len(X_cat)}")

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)

    y_test_cat, y_pred_cat = train_task_head(X_cat, y_cat, "VPN Category", use_smote=False)
    print(classification_report(y_test_cat, y_pred_cat, target_names=le_cat.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 3: APPLICATION TASK (Top 6 VPN Apps)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN APPLICATION (Top 6)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]

    app_mask = df_labels['Application'].isin(target_apps)
    X_app = X_final_all[app_mask]
    y_app_raw = df_labels.loc[app_mask, 'Application']

    print(f"Top 6 App Samples: {len(X_app)}")

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)

    y_test_app, y_pred_app = train_task_head(X_app, y_app, "VPN Top 6 Apps", use_smote=False)
    print(classification_report(y_test_app, y_pred_app, target_names=le_app.classes_, digits=4))

if __name__ == "__main__":
    main()

--- 0. Loading & Preprocessing ---
Loaded View 1: (12555, 10, 784)
Loaded View 2: (12555, 137)
Reconstructing Labels (VPN_Prefixing)...
Loading Encoder: /content/drive/MyDrive/1 Skripsi/27jan/v3FULL_HMVCL_Encoder.weights.h5
Weights loaded successfully.
Extracting features from Raw Payload...
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Final Fused Data Shape: (12555, 265)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 265)
    Train: 2511 | Test: 10044
    >>> Binary Task Macro F1: 0.9448
              precision    recall  f1-score   support

     Non-VPN     0.9852    0.9649    0.9749      7843
         VPN     0.8836    0.9482    0.9147      2201

    accuracy                         0.9613     10044
   macro avg     0.9344    0.9566    0.9448     10044
weighted avg     0.9629    0.9613    0.9618     10044


 EXPERIMENT 2: VPN CATEGORY CLASSIFICATION
VPN Samples: 2751

>>> Starting Task: VPN Category
    Data Shape: (2751, 265)
    Train: 550 | Te

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input Data (Ensure these point to your CLEAN, NEW files)
X_VIEW1_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view1.npy") # Raw Payload (N, 10, 784)
X_VIEW2_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view2.npy") # Stats (N, 137)
Y_LABELS_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Pre-trained Weights
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "v3FULL_HMVCL_Encoder.weights.h5")

# Training Config
LABEL_PERCENTAGE = 0.05 # 20% labeled data

# --- 1. Architecture Definition (MUST MATCH PRE-TRAINING) ---
def get_cnn_encoder(input_shape=(10, 784)):
    inputs = layers.Input(shape=input_shape)

    # 1. Reshape: Treat 10 packets * 784 bytes as one long sequence
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs) # (7840, 1)

    # 2. CNN Block (Exact same layers as Pre-training)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    # Flatten -> Dense
    x = layers.Flatten()(x)

    h = layers.Dense(128, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)

    # We return [h, z] to match the weights structure
    return Model(inputs, [h, z], name="CNN_Encoder")

# --- 2. Generic Trainer Helper ---
def train_task_head(X_subset, y_subset, task_name, use_smote=False):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    # Stratified Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # SMOTE (Imbalance Handling)
    if use_smote:
        class_counts = np.bincount(y_train)
        if np.min(class_counts) < 2:
            print("    ! Warning: Rare classes detected. Skipping SMOTE.")
            X_train_bal, y_train_bal = X_train, y_train
        else:
            print(f"    Applying SMOTE...")
            smote = SMOTE(k_neighbors=1, random_state=42)
            X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    # Determine Objective & Metric
    num_classes = len(np.unique(y_train_bal))
    if num_classes == 2:
        objective = 'binary:logistic'
        eval_metric = 'logloss'
    else:
        objective = 'multi:softprob'
        eval_metric = 'mlogloss'

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        eval_metric=eval_metric,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train_bal, y_train_bal, sample_weight=sample_weights)

    # Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    >>> {task_name} Macro F1: {f1:.4f}")

    return y_test, y_pred

def main():
    # ==========================================
    # 0. GLOBAL SETUP
    # ==========================================
    print("--- 0. Loading & Preprocessing ---")

    # Load Data
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, 137)

    print(f"Loaded View 1: {X_view1.shape}")
    print(f"Loaded View 2: {X_view2.shape}")

    df_raw = pd.read_csv(Y_LABELS_PATH)

    # Normalize Stats
    X_view2 = (X_view2 - np.mean(X_view2, axis=0)) / (np.std(X_view2, axis=0) + 1e-8)

    # --- LABEL RECONSTRUCTION LOGIC ---
    print("Reconstructing Labels (VPN_Prefixing)...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print("Error: Could not find 'application' or 'category' columns in CSV.")
        return

    final_traffic = []
    final_category = []
    final_app = []

    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix = "VPN"
            traffic_type = "VPN"
        else:
            prefix = "NonVPN"
            traffic_type = "Non-VPN"

        final_traffic.append(traffic_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # --- FEATURE FUSION ---
    print(f"Loading Encoder: {ENCODER_WEIGHTS_PATH}")

    # 1. Initialize Correct Model Architecture
    cnn_encoder = get_cnn_encoder(input_shape=(10, 784))

    # 2. Load Weights (This should now work perfectly)
    try:
        cnn_encoder.load_weights(ENCODER_WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"CRITICAL ERROR LOADING WEIGHTS: {e}")
        return

    # 3. Create Feature Extractor (Output is 'h', the representation)
    extractor = Model(inputs=cnn_encoder.input, outputs=cnn_encoder.outputs[0])

    print("Extracting features from Raw Payload...")
    h_features = extractor.predict(X_view1, batch_size=128, verbose=1)

    # 4. Concatenate: Stats + Learned Payload Embeddings
    X_final_all = np.concatenate([X_view2, h_features], axis=1)

    # --- CLEAN NaNs ---
    if np.isnan(X_final_all).any() or np.isinf(X_final_all).any():
        print("   ! Found NaNs/Infs in fused data. Cleaning...")
        X_final_all = np.nan_to_num(X_final_all, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Final Fused Data Shape: {X_final_all.shape}")


    # ==========================================
    # EXPERIMENT 1: BINARY TASK (Full Data)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])

    y_test_bin, y_pred_bin = train_task_head(X_final_all, y_bin, "Binary Task", use_smote=False)
    print(classification_report(y_test_bin, y_pred_bin, target_names=le_bin.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 2: CATEGORY TASK (VPN Only)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY CLASSIFICATION")
    print("="*40)

    vpn_mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final_all[vpn_mask]
    y_cat_raw = df_labels.loc[vpn_mask, 'Category']

    print(f"VPN Samples: {len(X_cat)}")

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)

    y_test_cat, y_pred_cat = train_task_head(X_cat, y_cat, "VPN Category", use_smote=False)
    print(classification_report(y_test_cat, y_pred_cat, target_names=le_cat.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 3: APPLICATION TASK (Top 6 VPN Apps)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN APPLICATION (Top 6)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]

    app_mask = df_labels['Application'].isin(target_apps)
    X_app = X_final_all[app_mask]
    y_app_raw = df_labels.loc[app_mask, 'Application']

    print(f"Top 6 App Samples: {len(X_app)}")

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)

    y_test_app, y_pred_app = train_task_head(X_app, y_app, "VPN Top 6 Apps", use_smote=False)
    print(classification_report(y_test_app, y_pred_app, target_names=le_app.classes_, digits=4))

if __name__ == "__main__":
    main()

--- 0. Loading & Preprocessing ---
Loaded View 1: (12555, 10, 784)
Loaded View 2: (12555, 137)
Reconstructing Labels (VPN_Prefixing)...
Loading Encoder: /content/drive/MyDrive/1 Skripsi/27jan/v3FULL_HMVCL_Encoder.weights.h5
Weights loaded successfully.
Extracting features from Raw Payload...
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Final Fused Data Shape: (12555, 265)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 265)
    Train: 627 | Test: 11928
    >>> Binary Task Macro F1: 0.8982
              precision    recall  f1-score   support

     Non-VPN     0.9616    0.9469    0.9542      9314
         VPN     0.8205    0.8653    0.8423      2614

    accuracy                         0.9290     11928
   macro avg     0.8910    0.9061    0.8982     11928
weighted avg     0.9307    0.9290    0.9297     11928


 EXPERIMENT 2: VPN CATEGORY CLASSIFICATION
VPN Samples: 2751

>>> Starting Task: VPN Category
    Data Shape: (2751, 265)
    Train: 137 | Tes

In [5]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input Data (Ensure these point to your CLEAN, NEW files)
X_VIEW1_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view1.npy") # Raw Payload (N, 10, 784)
X_VIEW2_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view2.npy") # Stats (N, 137)
Y_LABELS_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Pre-trained Weights
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "v3FULL_HMVCL_Encoder.weights.h5")

# Training Config
LABEL_PERCENTAGE = 0.10 # 20% labeled data

# --- 1. Architecture Definition (MUST MATCH PRE-TRAINING) ---
def get_cnn_encoder(input_shape=(10, 784)):
    inputs = layers.Input(shape=input_shape)

    # 1. Reshape: Treat 10 packets * 784 bytes as one long sequence
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs) # (7840, 1)

    # 2. CNN Block (Exact same layers as Pre-training)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    # Flatten -> Dense
    x = layers.Flatten()(x)

    h = layers.Dense(128, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)

    # We return [h, z] to match the weights structure
    return Model(inputs, [h, z], name="CNN_Encoder")

# --- 2. Generic Trainer Helper ---
def train_task_head(X_subset, y_subset, task_name, use_smote=False):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    # Stratified Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # SMOTE (Imbalance Handling)
    if use_smote:
        class_counts = np.bincount(y_train)
        if np.min(class_counts) < 2:
            print("    ! Warning: Rare classes detected. Skipping SMOTE.")
            X_train_bal, y_train_bal = X_train, y_train
        else:
            print(f"    Applying SMOTE...")
            smote = SMOTE(k_neighbors=1, random_state=42)
            X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    # Determine Objective & Metric
    num_classes = len(np.unique(y_train_bal))
    if num_classes == 2:
        objective = 'binary:logistic'
        eval_metric = 'logloss'
    else:
        objective = 'multi:softprob'
        eval_metric = 'mlogloss'

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        eval_metric=eval_metric,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train_bal, y_train_bal, sample_weight=sample_weights)

    # Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    >>> {task_name} Macro F1: {f1:.4f}")

    return y_test, y_pred

def main():
    # ==========================================
    # 0. GLOBAL SETUP
    # ==========================================
    print("--- 0. Loading & Preprocessing ---")

    # Load Data
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, 137)

    print(f"Loaded View 1: {X_view1.shape}")
    print(f"Loaded View 2: {X_view2.shape}")

    df_raw = pd.read_csv(Y_LABELS_PATH)

    # Normalize Stats
    X_view2 = (X_view2 - np.mean(X_view2, axis=0)) / (np.std(X_view2, axis=0) + 1e-8)

    # --- LABEL RECONSTRUCTION LOGIC ---
    print("Reconstructing Labels (VPN_Prefixing)...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print("Error: Could not find 'application' or 'category' columns in CSV.")
        return

    final_traffic = []
    final_category = []
    final_app = []

    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix = "VPN"
            traffic_type = "VPN"
        else:
            prefix = "NonVPN"
            traffic_type = "Non-VPN"

        final_traffic.append(traffic_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # --- FEATURE FUSION ---
    print(f"Loading Encoder: {ENCODER_WEIGHTS_PATH}")

    # 1. Initialize Correct Model Architecture
    cnn_encoder = get_cnn_encoder(input_shape=(10, 784))

    # 2. Load Weights (This should now work perfectly)
    try:
        cnn_encoder.load_weights(ENCODER_WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"CRITICAL ERROR LOADING WEIGHTS: {e}")
        return

    # 3. Create Feature Extractor (Output is 'h', the representation)
    extractor = Model(inputs=cnn_encoder.input, outputs=cnn_encoder.outputs[0])

    print("Extracting features from Raw Payload...")
    h_features = extractor.predict(X_view1, batch_size=128, verbose=1)

    # 4. Concatenate: Stats + Learned Payload Embeddings
    X_final_all = np.concatenate([X_view2, h_features], axis=1)

    # --- CLEAN NaNs ---
    if np.isnan(X_final_all).any() or np.isinf(X_final_all).any():
        print("   ! Found NaNs/Infs in fused data. Cleaning...")
        X_final_all = np.nan_to_num(X_final_all, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Final Fused Data Shape: {X_final_all.shape}")


    # ==========================================
    # EXPERIMENT 1: BINARY TASK (Full Data)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])

    y_test_bin, y_pred_bin = train_task_head(X_final_all, y_bin, "Binary Task", use_smote=False)
    print(classification_report(y_test_bin, y_pred_bin, target_names=le_bin.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 2: CATEGORY TASK (VPN Only)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY CLASSIFICATION")
    print("="*40)

    vpn_mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final_all[vpn_mask]
    y_cat_raw = df_labels.loc[vpn_mask, 'Category']

    print(f"VPN Samples: {len(X_cat)}")

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)

    y_test_cat, y_pred_cat = train_task_head(X_cat, y_cat, "VPN Category", use_smote=False)
    print(classification_report(y_test_cat, y_pred_cat, target_names=le_cat.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 3: APPLICATION TASK (Top 6 VPN Apps)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN APPLICATION (Top 6)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]

    app_mask = df_labels['Application'].isin(target_apps)
    X_app = X_final_all[app_mask]
    y_app_raw = df_labels.loc[app_mask, 'Application']

    print(f"Top 6 App Samples: {len(X_app)}")

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)

    y_test_app, y_pred_app = train_task_head(X_app, y_app, "VPN Top 6 Apps", use_smote=False)
    print(classification_report(y_test_app, y_pred_app, target_names=le_app.classes_, digits=4))

if __name__ == "__main__":
    main()

--- 0. Loading & Preprocessing ---
Loaded View 1: (12555, 10, 784)
Loaded View 2: (12555, 137)
Reconstructing Labels (VPN_Prefixing)...
Loading Encoder: /content/drive/MyDrive/1 Skripsi/27jan/v3FULL_HMVCL_Encoder.weights.h5
Weights loaded successfully.
Extracting features from Raw Payload...
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Final Fused Data Shape: (12555, 265)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 265)
    Train: 1255 | Test: 11300
    >>> Binary Task Macro F1: 0.9218
              precision    recall  f1-score   support

     Non-VPN     0.9747    0.9544    0.9644      8824
         VPN     0.8488    0.9116    0.8791      2476

    accuracy                         0.9450     11300
   macro avg     0.9117    0.9330    0.9218     11300
weighted avg     0.9471    0.9450    0.9457     11300


 EXPERIMENT 2: VPN CATEGORY CLASSIFICATION
VPN Samples: 2751

>>> Starting Task: VPN Category
    Data Shape: (2751, 265)
    Train: 275 | Te

In [6]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras import layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from imblearn.over_sampling import SMOTE

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input Data (Ensure these point to your CLEAN, NEW files)
X_VIEW1_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view1.npy") # Raw Payload (N, 10, 784)
X_VIEW2_PATH = os.path.join(BASE_PATH, "v3PRETRAIN_X_view2.npy") # Stats (N, 137)
Y_LABELS_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Pre-trained Weights
ENCODER_WEIGHTS_PATH = os.path.join(BASE_PATH, "v3FULL_HMVCL_Encoder.weights.h5")

# Training Config
LABEL_PERCENTAGE = 0.30 # 20% labeled data

# --- 1. Architecture Definition (MUST MATCH PRE-TRAINING) ---
def get_cnn_encoder(input_shape=(10, 784)):
    inputs = layers.Input(shape=input_shape)

    # 1. Reshape: Treat 10 packets * 784 bytes as one long sequence
    x = layers.Reshape((input_shape[0] * input_shape[1], 1))(inputs) # (7840, 1)

    # 2. CNN Block (Exact same layers as Pre-training)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(4)(x)

    # Flatten -> Dense
    x = layers.Flatten()(x)

    h = layers.Dense(128, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu', name="projection")(h)

    # We return [h, z] to match the weights structure
    return Model(inputs, [h, z], name="CNN_Encoder")

# --- 2. Generic Trainer Helper ---
def train_task_head(X_subset, y_subset, task_name, use_smote=False):
    print(f"\n>>> Starting Task: {task_name}")
    print(f"    Data Shape: {X_subset.shape}")

    # Stratified Split
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y_subset,
        train_size=LABEL_PERCENTAGE,
        random_state=42,
        stratify=y_subset
    )
    print(f"    Train: {len(X_train)} | Test: {len(X_test)}")

    # SMOTE (Imbalance Handling)
    if use_smote:
        class_counts = np.bincount(y_train)
        if np.min(class_counts) < 2:
            print("    ! Warning: Rare classes detected. Skipping SMOTE.")
            X_train_bal, y_train_bal = X_train, y_train
        else:
            print(f"    Applying SMOTE...")
            smote = SMOTE(k_neighbors=1, random_state=42)
            X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
    else:
        X_train_bal, y_train_bal = X_train, y_train

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    # Determine Objective & Metric
    num_classes = len(np.unique(y_train_bal))
    if num_classes == 2:
        objective = 'binary:logistic'
        eval_metric = 'logloss'
    else:
        objective = 'multi:softprob'
        eval_metric = 'mlogloss'

    # XGBoost Training
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_bal)

    clf = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective=objective,
        eval_metric=eval_metric,
        random_state=42,
        n_jobs=-1
    )

    clf.fit(X_train_bal, y_train_bal, sample_weight=sample_weights)

    # Evaluate
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    print(f"    >>> {task_name} Macro F1: {f1:.4f}")

    return y_test, y_pred

def main():
    # ==========================================
    # 0. GLOBAL SETUP
    # ==========================================
    print("--- 0. Loading & Preprocessing ---")

    # Load Data
    X_view1 = np.load(X_VIEW1_PATH).astype('float32') # (N, 10, 784)
    X_view2 = np.load(X_VIEW2_PATH).astype('float32') # (N, 137)

    print(f"Loaded View 1: {X_view1.shape}")
    print(f"Loaded View 2: {X_view2.shape}")

    df_raw = pd.read_csv(Y_LABELS_PATH)

    # Normalize Stats
    X_view2 = (X_view2 - np.mean(X_view2, axis=0)) / (np.std(X_view2, axis=0) + 1e-8)

    # --- LABEL RECONSTRUCTION LOGIC ---
    print("Reconstructing Labels (VPN_Prefixing)...")
    app_col = next((c for c in df_raw.columns if c.endswith('application')), None)
    cat_col = next((c for c in df_raw.columns if c.endswith('category')), None)

    if not app_col or not cat_col:
        print("Error: Could not find 'application' or 'category' columns in CSV.")
        return

    final_traffic = []
    final_category = []
    final_app = []

    for idx, row in df_raw.iterrows():
        fname = str(row['filename']).lower()
        raw_app = str(row[app_col])
        raw_cat = str(row[cat_col])

        if "vpn" in fname:
            prefix = "VPN"
            traffic_type = "VPN"
        else:
            prefix = "NonVPN"
            traffic_type = "Non-VPN"

        final_traffic.append(traffic_type)
        final_category.append(f"{prefix}_{raw_cat}")
        final_app.append(f"{prefix}_{raw_app}")

    df_labels = pd.DataFrame({
        'Traffic_Type': final_traffic,
        'Category': final_category,
        'Application': final_app
    })

    # --- FEATURE FUSION ---
    print(f"Loading Encoder: {ENCODER_WEIGHTS_PATH}")

    # 1. Initialize Correct Model Architecture
    cnn_encoder = get_cnn_encoder(input_shape=(10, 784))

    # 2. Load Weights (This should now work perfectly)
    try:
        cnn_encoder.load_weights(ENCODER_WEIGHTS_PATH)
        print("Weights loaded successfully.")
    except Exception as e:
        print(f"CRITICAL ERROR LOADING WEIGHTS: {e}")
        return

    # 3. Create Feature Extractor (Output is 'h', the representation)
    extractor = Model(inputs=cnn_encoder.input, outputs=cnn_encoder.outputs[0])

    print("Extracting features from Raw Payload...")
    h_features = extractor.predict(X_view1, batch_size=128, verbose=1)

    # 4. Concatenate: Stats + Learned Payload Embeddings
    X_final_all = np.concatenate([X_view2, h_features], axis=1)

    # --- CLEAN NaNs ---
    if np.isnan(X_final_all).any() or np.isinf(X_final_all).any():
        print("   ! Found NaNs/Infs in fused data. Cleaning...")
        X_final_all = np.nan_to_num(X_final_all, nan=0.0, posinf=0.0, neginf=0.0)

    print(f"Final Fused Data Shape: {X_final_all.shape}")


    # ==========================================
    # EXPERIMENT 1: BINARY TASK (Full Data)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 1: BINARY DETECTION")
    print("="*40)

    le_bin = LabelEncoder()
    y_bin = le_bin.fit_transform(df_labels['Traffic_Type'])

    y_test_bin, y_pred_bin = train_task_head(X_final_all, y_bin, "Binary Task", use_smote=False)
    print(classification_report(y_test_bin, y_pred_bin, target_names=le_bin.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 2: CATEGORY TASK (VPN Only)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 2: VPN CATEGORY CLASSIFICATION")
    print("="*40)

    vpn_mask = df_labels['Traffic_Type'] == 'VPN'
    X_cat = X_final_all[vpn_mask]
    y_cat_raw = df_labels.loc[vpn_mask, 'Category']

    print(f"VPN Samples: {len(X_cat)}")

    le_cat = LabelEncoder()
    y_cat = le_cat.fit_transform(y_cat_raw)

    y_test_cat, y_pred_cat = train_task_head(X_cat, y_cat, "VPN Category", use_smote=False)
    print(classification_report(y_test_cat, y_pred_cat, target_names=le_cat.classes_, digits=4))


    # ==========================================
    # EXPERIMENT 3: APPLICATION TASK (Top 6 VPN Apps)
    # ==========================================
    print("\n" + "="*40)
    print(" EXPERIMENT 3: VPN APPLICATION (Top 6)")
    print("="*40)

    target_apps = [
        'VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout',
        'VPN_Facebook', 'VPN_YouTube', 'VPN_Email'
    ]

    app_mask = df_labels['Application'].isin(target_apps)
    X_app = X_final_all[app_mask]
    y_app_raw = df_labels.loc[app_mask, 'Application']

    print(f"Top 6 App Samples: {len(X_app)}")

    le_app = LabelEncoder()
    y_app = le_app.fit_transform(y_app_raw)

    y_test_app, y_pred_app = train_task_head(X_app, y_app, "VPN Top 6 Apps", use_smote=False)
    print(classification_report(y_test_app, y_pred_app, target_names=le_app.classes_, digits=4))

if __name__ == "__main__":
    main()

--- 0. Loading & Preprocessing ---
Loaded View 1: (12555, 10, 784)
Loaded View 2: (12555, 137)
Reconstructing Labels (VPN_Prefixing)...
Loading Encoder: /content/drive/MyDrive/1 Skripsi/27jan/v3FULL_HMVCL_Encoder.weights.h5
Weights loaded successfully.
Extracting features from Raw Payload...
99/99 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
Final Fused Data Shape: (12555, 265)

 EXPERIMENT 1: BINARY DETECTION

>>> Starting Task: Binary Task
    Data Shape: (12555, 265)
    Train: 3766 | Test: 8789
    >>> Binary Task Macro F1: 0.9493
              precision    recall  f1-score   support

     Non-VPN     0.9882    0.9658    0.9769      6863
         VPN     0.8871    0.9590    0.9217      1926

    accuracy                         0.9643      8789
   macro avg     0.9377    0.9624    0.9493      8789
weighted avg     0.9661    0.9643    0.9648      8789


 EXPERIMENT 2: VPN CATEGORY CLASSIFICATION
VPN Samples: 2751

>>> Starting Task: VPN Category
    Data Shape: (2751, 265)
    Train: 825 | Tes